# lawforge-harvest: Kimina-Prover-RL-1.7B pass@K proof harvester

Generates K diverse Lean 4 proof candidates per SAIR equational-theory problem (train + dev + hard2 + hard3 splits). Saves raw candidates to `/kaggle/working/harvested.jsonl` for local judging.

Runtime estimate (T4 fp16, K=32, 1669 problems, batch=8): ~6-8h.

Output schema per row:
```json
{"id": str, "split": str, "eq1": str, "eq2": str, "label": str|null,
 "candidates": [str, ...]}
```

In [ ]:
!pip -q install 'vllm==0.22.0' 'transformers>=4.51' accelerate

In [ ]:
import os, subprocess, sys

REPO = "/kaggle/working/lawforge"
if not os.path.isdir(REPO):
    subprocess.check_call(
        ["git", "clone", "--depth", "1", "https://github.com/PAMF2/lawforge.git", REPO]
    )
sys.path.insert(0, REPO)
print(
    "repo HEAD:",
    subprocess.check_output(["git", "-C", REPO, "log", "-1", "--oneline"])
    .decode()
    .strip(),
)

In [ ]:
import os
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

MODEL = os.environ.get("LAWFORGE_HARVEST_MODEL", "AI-MO/Kimina-Prover-RL-1.7B")
K = int(os.environ.get("LAWFORGE_HARVEST_K", "32"))
MAX_TOKENS = int(os.environ.get("LAWFORGE_HARVEST_MAX_TOKENS", "4096"))
TEMP = float(os.environ.get("LAWFORGE_HARVEST_TEMP", "0.6"))
TOP_P = float(os.environ.get("LAWFORGE_HARVEST_TOP_P", "0.95"))
DTYPE = os.environ.get("LAWFORGE_HARVEST_DTYPE", "float16")

tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
llm = LLM(
    model=MODEL,
    dtype=DTYPE,
    max_model_len=8192,
    gpu_memory_utilization=0.90,
    trust_remote_code=True,
)
params = SamplingParams(n=K, temperature=TEMP, top_p=TOP_P, max_tokens=MAX_TOKENS)
print(
    f"model={MODEL} K={K} temp={TEMP} top_p={TOP_P} max_tokens={MAX_TOKENS} dtype={DTYPE}"
)

In [ ]:
import json
from pathlib import Path

INPUTS = Path(f'{REPO}/kaggle/harvest/inputs')
SPLITS = ['train_split', 'dev_split', 'hard2_test', 'hard3_test']
LIMIT = int(os.environ.get('LAWFORGE_HARVEST_LIMIT', '0'))  # 0 = all

problems = []
for s in SPLITS:
    path = INPUTS / f'{s}.jsonl'
    with path.open() as f:
        for line in f:
            row = json.loads(line)
            row['_split'] = s
            problems.append(row)
if LIMIT > 0:
    problems = problems[:LIMIT]
    print(f'SMOKE MODE: capped at {LIMIT} problems')
print(f'loaded {len(problems)} problems across {len(SPLITS)} splits')

In [ ]:
# Kimina-Prover-RL-1.7B official prompt format (per HF model card).
# System prompt verbatim, user message uses "# Problem:" + "# Formal statement:"
# fenced lean4 block. Chat template applied via Qwen3 tokenizer.

SYSTEM = 'You are an expert in mathematics and proving theorems in Lean 4.'

LEAN_STATEMENT_TPL = (
    'import Mathlib\n'
    'import Aesop\n'
    'set_option maxHeartbeats 400000\n'
    'class Magma (G : Type) where\n'
    '  op : G → G → G\n'
    'infixl:70 " ◇ " => Magma.op\n\n'
    'theorem sair_implication\n'
    '    (G : Type) [inst : Magma G]\n'
    '    (h : ∀ x y z w u : G, {eq1})\n'
    '    : ∀ x y z w u : G, {eq2} := by\n'
    '  sorry'
)


def to_diamond(s: str) -> str:
    """Convert SAIR magma operator `*` to `◇` (U+25C7) so the embedded Lean
    statement typechecks against the Magma class declared in the template.
    Mirrors lean.judge._to_diamond. Idempotent on `◇`."""
    return s.replace('*', '◇') if s else s


def build_prompt(p: dict) -> str:
    eq1 = to_diamond(p.get('equation1') or p.get('hypothesis', ''))
    eq2 = to_diamond(p.get('equation2') or p.get('goal', ''))
    statement = LEAN_STATEMENT_TPL.format(eq1=eq1, eq2=eq2)
    goal_str = (
        f"Prove that in a magma G with operator ◇, the hypothesis "
        f"`∀ x y z w u, {eq1}` implies `∀ x y z w u, {eq2}`. "
        f"Use only basic Lean 4 tactics that do not require Mathlib "
        f"lemma references."
    )
    user = (
        "Think about and solve the following problem step by step in Lean 4.\n"
        f"# Problem: {goal_str}\n"
        "# Formal statement:\n"
        "```lean4\n"
        f"{statement}\n"
        "```"
    )
    return tokenizer.apply_chat_template(
        [{'role': 'system', 'content': SYSTEM},
         {'role': 'user', 'content': user}],
        tokenize=False, add_generation_prompt=True,
    )


print('sample prompt for problem 0 (last 800 chars):')
print(build_prompt(problems[0])[-800:])

In [ ]:
import time

OUT = Path("/kaggle/working/harvested.jsonl")
BATCH = int(os.environ.get("LAWFORGE_HARVEST_BATCH", "8"))
TACTIC_KEYS = (
    "intro",
    "rw",
    "apply",
    "have",
    "exact",
    "simp",
    "aesop",
    "calc",
    "refine",
    "symm",
    "cases",
    "rfl",
    "decide",
    "assumption",
    "nth_rewrite",
    "repeat",
    "fun ",
)


def looks_like_proof(text: str) -> bool:
    s = text.strip()
    if len(s) < 4:
        return False
    low = s.lower()
    return any(k in low for k in TACTIC_KEYS)


t0 = time.time()
with OUT.open("w") as out:
    for i in range(0, len(problems), BATCH):
        batch = problems[i : i + BATCH]
        prompts = [build_prompt(p) for p in batch]
        results = llm.generate(prompts, params, use_tqdm=False)
        for p, r in zip(batch, results):
            cands = [o.text for o in r.outputs if looks_like_proof(o.text)]
            out.write(
                json.dumps(
                    {
                        "id": p.get("id", ""),
                        "split": p.get("_split", ""),
                        "eq1": p.get("equation1") or p.get("hypothesis", ""),
                        "eq2": p.get("equation2") or p.get("goal", ""),
                        "label": p.get("label"),
                        "candidates": cands,
                    }
                )
                + "\n"
            )
            out.flush()
        done = i + len(batch)
        if done % (BATCH * 4) == 0 or done == len(problems):
            elapsed = time.time() - t0
            rate = done / max(1, elapsed)
            eta = (len(problems) - done) / max(0.01, rate)
            print(
                f"[{done}/{len(problems)}] {elapsed:.0f}s elapsed, "
                f"{rate:.2f} prob/s, ETA {eta:.0f}s"
            )
print("harvest done in", time.time() - t0, "s")
print("output bytes:", OUT.stat().st_size)

In [ ]:
!ls -lh /kaggle/working/harvested.jsonl
!wc -l /kaggle/working/harvested.jsonl
!head -1 /kaggle/working/harvested.jsonl | python3 -c 'import json,sys; r=json.loads(sys.stdin.read()); print("id=", r["id"], "candidates=", len(r["candidates"]), "first=", r["candidates"][0][:200] if r["candidates"] else "")'